# Search and highlight — `repo.search()` + `widget.highlight_match()`

This notebook demonstrates finding exact sequences in a pangenome graph and
highlighting the matched nodes directly in the interactive viewer.

**Workflow:**
1. Import the Anderson promoter GFA
2. Open the interactive graph widget
3. Search without an index — `search()` falls back to a full scan
4. Build a seed index, search again with the same call — now accelerated
5. Highlight multiple motifs in different colours
6. Search across multiple block groups

## Setup

We import the [Anderson promoter collection](http://parts.igem.org/Promoters/Catalog/Anderson)
as a GFA — this gives us a proper pangenome graph with shared nodes and junctions.

In [ ]:
import pathlib
import tempfile

import gen

REPO_ROOT = pathlib.Path(gen.__file__).parents[3]
GFA = REPO_ROOT / "fixtures" / "anderson_promoters.gfa"
assert GFA.exists(), f"Fixture not found: {GFA}"

WORK_DIR = pathlib.Path(tempfile.mkdtemp(prefix="gen-search-"))
print(f"Working in {WORK_DIR}")

repo = gen.Repository(str(WORK_DIR))
repo.import_gfa(str(GFA), 'anderson', 'pooled')
bgs = repo.get_block_groups_by_collection('anderson')
print(f"{len(bgs)} block group(s) imported")
for b in bgs:
    print(f"  {b}")

## Open the graph widget

Store the widget — calls to `highlight_match()` and `clear_highlights()` update
it **in place** in this cell without re-running it.

In [ ]:
bg = bgs[0]
widget = bg.plot(detail="full")

## Search without an index

`repo.search()` works out of the box — no index required.  When no `.bin` file
is found for a block group it falls back to a full graph scan automatically.

The query `"ctagctcagt"` is the conserved **−10 box** core present in all
Anderson promoters.

In [ ]:
QUERY = "ctagctcagt"

matches = repo.search(bg, QUERY)
print(f"Found {len(matches)} match(es) for {QUERY!r} (full scan)")

widget.clear_highlights()
for m in matches:
    widget.highlight_match(m)
    print(" ", m)

## Build a seed index

`repo.build_index()` saves a junction-aware k-mer index to
`.gen/search_index/<id>.bin`.  Once it exists, `search()` picks it up
automatically — the call is identical, just faster.

- Omit `bgs` to index all block groups at once.
- `k` defaults to 16; smaller values work fine for short sequences.
- `bg.build_index()` indexes a single block group in place.

In [ ]:
repo.build_index(k=8)
print("Index built.")

## Search again — now with the index

Exactly the same call.  The index is loaded from disk automatically and the
search is seed-extended rather than a full scan.

In [ ]:
matches = repo.search(bg, QUERY)
print(f"Found {len(matches)} match(es) for {QUERY!r} (indexed)")

widget.clear_highlights()
for m in matches:
    widget.highlight_match(m)
    print(" ", m)

## Highlight multiple motifs in different colours

`highlight_match()` accepts an optional `color` — a named ratatui colour
(`"yellow"`, `"cyan"`, `"red"`, `"green"`, `"magenta"`, …) or a CSS hex
string like `"#ffaa00"`.  Stack several calls to paint multiple motifs at once.

In [ ]:
MOTIFS = [
    ("ttgac",      "yellow"),  # −35 box
    ("ctagctcagt", "cyan"),    # −10 box
]

widget.clear_highlights()
for query, color in MOTIFS:
    hits = repo.search(bg, query)
    for m in hits:
        widget.highlight_match(m, color)
    print(f"{query!r:15s}  {len(hits)} hit(s)  [{color}]")

## Search and highlight across multiple block groups

Open a second widget for another block group and highlight the same motif there.
All indices are already on disk from the `build_index()` call above.

In [ ]:
MINUS_35 = "ttgac"

hits_by_bg = [
    (b, found)
    for b in bgs
    if (found := repo.search(b, MINUS_35))
]

print(f"'{MINUS_35}' found in {len(hits_by_bg)} / {len(bgs)} block group(s)")
for b, hits in hits_by_bg:
    print(f"  {b.name:15s}  {len(hits)} hit(s)")

if len(hits_by_bg) > 1:
    bg2, hits2 = hits_by_bg[1]
    widget2 = bg2.plot(detail="full")
    for m in hits2:
        widget2.highlight_match(m, "yellow")
    print(f"\nOpened {bg2.name!r} with {len(hits2)} highlight(s)")

## Clear the index

`repo.clear_index()` removes all `.bin` files under `.gen/search_index/`.
Pass `bgs=[...]` to clear only specific entries.  `bg.clear_index()` removes
just that one.

In [ ]:
repo.clear_index()
# or: bg.clear_index()
print("Index cleared.")

## Tips

**Index lifetime** — the index persists on disk between sessions.  Build once
after import; rebuild only if the graph changes.

**Cross-junction matches** — when a query spans a node boundary, `highlight_match()`
paints all nodes in the locus.  Check `len(m.nodes) > 1` to detect these.

**Clearing highlights** — call `widget.clear_highlights()` before re-searching
to avoid accumulating stale highlight layers.